# GRPO v3 on Colab — Grounded RAG for Mental Health

CLAUDE.md Act 2 step 10, second iteration. Adds a **per-sentence citation_compliance** term to the reward to sharpen the citation-repair signal that v2 only weakly moved.

**Reward formula:**
`R = g − 0.5·copy + 0.3·cite_r + 0.6·compliance − 0.4·abst·(g≥0.5)`

**What's new vs `grpo_colab.ipynb` (v1/v2):**
- Reuses the v2 100-question prompt pool (`data/grpo/prompts_v2.jsonl`) — no re-retrieval needed.
- Uses `configs/grpo_v3.yaml`.
- Adds **`save_steps=25`** so a disconnect loses at most ~25 steps, not a whole epoch.
- Adds a **background Drive mirror** that rsyncs checkpoints + reward trace every 2 min while training runs. Two lost-work incidents in v2 motivated this.
- Watches the new `comp=` column in the reward log. Should climb from ~0.1 to 0.5+ over the run if the sharper reward works.

**Runtime target:** A100 (~20 min training). Falls back cleanly on V100/T4.

## 1. Environment setup

**Before running:** in the left sidebar, click the 🔑 icon and add a secret named `COHERE_API_KEY` with your production key.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'NO GPU — switch runtime')

In [ ]:
REPO_URL = 'https://github.com/nightvoyager111/Grounded-RAG-via-RL-for-mental-health.git'
!rm -rf Grounded-RAG-via-RL-for-mental-health
!git clone {REPO_URL}
%cd Grounded-RAG-via-RL-for-mental-health

In [ ]:
!pip install -q cohere python-dotenv pyyaml transformers accelerate peft 'trl>=0.11' datasets sentencepiece

In [ ]:
from google.colab import userdata
import os
os.environ['COHERE_API_KEY'] = userdata.get('COHERE_API_KEY')
assert os.environ['COHERE_API_KEY'], 'COHERE_API_KEY secret is empty — set it in the 🔑 sidebar'

## 2. Restore artifacts from Drive

v3 depends on: (a) the DPO adapter (merged into the base before training), and (b) the v2 prompt file (`prompts_v2.jsonl`). Both should be on Drive from earlier runs. If either is missing, the fallback cells below regenerate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p checkpoints data/grpo

# DPO adapter (base + this gets merged before v3 LoRA is trained on top).
!cp -r /content/drive/MyDrive/grounded-rag-dpo/checkpoints/dpo checkpoints/
!ls checkpoints/dpo/adapter_config.json  # must exist

# v2 prompts (same 100 questions × retrieved passages).
!cp /content/drive/MyDrive/grounded-rag-dpo/grpo_artifacts/prompts_v2.jsonl data/grpo/ 2>/dev/null || echo 'prompts_v2.jsonl not on Drive — will regenerate'
!wc -l data/grpo/prompts_v2.jsonl 2>/dev/null || echo 'prompts_v2.jsonl missing'

### Fallback: regenerate prompts_v2.jsonl if it's not on Drive

Skip this cell if the previous cell showed `100 data/grpo/prompts_v2.jsonl`. If it showed 'missing', this cell regenerates from the existing expanded question set (~2–5 min Cohere calls, ~$0.05).

In [ ]:
import os
if not os.path.exists('data/grpo/prompts_v2.jsonl') or os.path.getsize('data/grpo/prompts_v2.jsonl') == 0:
    # Copy expanded_questions from Drive if we have it, else regenerate from corpus.
    !mkdir -p data/qa
    !cp /content/drive/MyDrive/grounded-rag-dpo/grpo_artifacts/expanded_questions.jsonl data/qa/ 2>/dev/null || python -m src.scripts.generate_questions --target 100
    !python -m src.scripts.prepare_grpo_prompts --grpo-config configs/grpo_v3.yaml
    !cp data/grpo/prompts_v2.jsonl /content/drive/MyDrive/grounded-rag-dpo/grpo_artifacts/
    !cp data/qa/expanded_questions.jsonl /content/drive/MyDrive/grounded-rag-dpo/grpo_artifacts/
print('prompts ready:', __import__('subprocess').run(['wc','-l','data/grpo/prompts_v2.jsonl'], capture_output=True, text=True).stdout)

## 3. Config overrides

In [ ]:
# Rewrite generation.yaml for CUDA + fp16 (matches DPO/v2 runs).
import yaml, pathlib
gen_path = pathlib.Path('configs/generation.yaml')
gen = yaml.safe_load(gen_path.read_text())
gen['device'] = 'cuda'
gen['dtype'] = 'float16'
gen_path.write_text(yaml.safe_dump(gen, sort_keys=False))
print(gen_path.read_text())

In [ ]:
# Safer save schedule: every 25 steps instead of per epoch. Combined with the
# Drive mirror below, a disconnect loses at most ~25 steps.
p = pathlib.Path('configs/grpo_v3.yaml')
c = yaml.safe_load(p.read_text())
c['save_strategy'] = 'steps'
c['save_steps'] = 25
p.write_text(yaml.safe_dump(c, sort_keys=False))
print('save_strategy:', c['save_strategy'], '| save_steps:', c['save_steps'])

## 4. Background Drive mirror

Runs in a daemon thread while training. Every 2 min, rsyncs `checkpoints/` and `data/grpo/` to Drive. If Colab disconnects mid-run, everything up to the last 2 min is safe on Drive.

In [ ]:
import subprocess, threading, time, os

DRIVE_BASE = '/content/drive/MyDrive/grounded-rag-dpo'
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/grpo_artifacts', exist_ok=True)

def _mirror_loop():
    while True:
        subprocess.run(['rsync', '-a', 'checkpoints/', f'{DRIVE_BASE}/checkpoints/'],
                       stderr=subprocess.DEVNULL)
        subprocess.run(['rsync', '-a', 'data/grpo/', f'{DRIVE_BASE}/grpo_artifacts/'],
                       stderr=subprocess.DEVNULL)
        time.sleep(120)

threading.Thread(target=_mirror_loop, daemon=True).start()
print('Drive mirror running (every 2 min).')

## 5. Train v3

Watch the `[reward step ####]` lines for the new `comp=` column. Should climb from ~0.1 (early) toward 0.4–0.6 (late) if the compliance reward is doing what we designed. `abst` should stay near 0 (ρ was bumped 0.3 → 0.4).

In [ ]:
!python -m src.scripts.train_grpo --grpo-config configs/grpo_v3.yaml

## 6. Final persist + evaluate

In [ ]:
# Belt + suspenders vs the background mirror.
!rsync -a checkpoints/grpo_v3 /content/drive/MyDrive/grounded-rag-dpo/checkpoints/
!rsync -a data/grpo/reward_trace_v3.jsonl /content/drive/MyDrive/grounded-rag-dpo/grpo_artifacts/

In [ ]:
!python -m src.scripts.evaluate_grpo \
    --grpo-config configs/grpo_v3.yaml \
    --grpo-adapter checkpoints/grpo_v3 \
    --output-dir src/results/grpo_v3

In [ ]:
!rsync -a src/results/grpo_v3 /content/drive/MyDrive/grounded-rag-dpo/results/
!cat src/results/grpo_v3/report.json

## 7. Per-step collapse/repair table from the reward trace

Same shape as v1/v2 tables, plus the new `comp` column. This is the artifact for CLAUDE.md step 10.

In [ ]:
import json, collections, statistics as st
buckets = collections.defaultdict(list)
with open('data/grpo/reward_trace_v3.jsonl') as f:
    for line in f:
        buckets[json.loads(line)['step']].append(json.loads(line))

def _mean(vals):
    v = [x for x in vals if x is not None]
    return st.mean(v) if v else float('nan')

print(f"{'step':>4} {'R':>6} {'g':>6} {'copy':>6} {'cite_r':>7} {'comp':>6} {'abst':>5}")
for step in sorted(buckets):
    rs = buckets[step]
    print(f"{step:>4} {_mean([r['reward'] for r in rs]):>6.3f} "
          f"{_mean([r['groundedness'] for r in rs]):>6.3f} "
          f"{_mean([r['copy_rate'] for r in rs]):>6.3f} "
          f"{_mean([r['citation_recall'] for r in rs]):>7.3f} "
          f"{_mean([r['citation_compliance'] for r in rs]):>6.3f} "
          f"{_mean([r['abstention'] for r in rs]):>5.2f}")